# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing all dataset entities by their `@id`s for accuracy and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print out the dataset name and description
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s from the dataset for further exploration.

In [ ]:
# List record sets with their @ids
print('Available record sets:')
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', 'No name')}")

# For illustration, print out record sets and a few sample fields for each
for record_set in dataset.record_sets:
    print(f"\nRecord set @id: {record_set['@id']}")
    print(f"  Name: {record_set.get('name', 'No name')}")
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field_id in fields:
        field = dataset.field_by_id(field_id)
        print(f"    - {getattr(field, '@id', str(field_id))}: {getattr(field, 'name', 'No field name')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set, field, and column references are by their `@id`.

In [ ]:
# Compile the list of available record_set @ids
record_set_ids = [recset['@id'] for recset in dataset.record_sets]
print('Loading record sets:', record_set_ids)

# Load each record set into a DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    try:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'Record set {record_set_id}: Loaded {len(df)} records.')
    except Exception as e:
        print(f'Could not load record set {record_set_id} ({e})')

# For demonstration, select the first record_set for EDA
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
print(f"\nFields/Columns in '{main_record_set_id}':")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping using field IDs. Please ensure you reference columns by their `@id` in all code.

In [ ]:
# For demonstration, choose meaningful numeric and group fields by @id
# (Replace these with valid field @ids from the previous 'print(df.columns.tolist())' output)
# Example: 'cr:field/protein_coverage', 'cr:field/protein_abundance', etc.

# Assign IDs for columns you wish to explore
numeric_field_id = df.columns[0]  # Change if your numeric field is different
group_field_id = df.columns[1] if len(df.columns) > 1 else df.columns[0]  # Example: a group ID

print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

# Convert the field to numeric if possible (ignore errors if values are non-numeric)
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filtering: Select records where the numeric field is greater than a threshold
threshold = df[numeric_field_id].mean() if df[numeric_field_id].dtype.kind in 'fi' else 0

filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical/group field if it exists
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field_id} (mean of numeric columns):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the numeric field or its relationship to the grouping field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field_id].dropna(), bins=20)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a group field is present, show boxplot for numeric by group
if group_field_id in df.columns:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xticks(rotation=90)
    plt.show()

## 6. Conclusion
In this notebook, you loaded the FAIR^2 dataset defined by a Croissant schema, explored its record sets and fields using their `@id`s, extracted tabular data, and performed initial filtering, normalization, grouping, and visualizations. To extend this workflow, consult the Croissant file for precise field semantics and further tailor the exploration to your research questions.
